In [ ]:
# ============================================================
# GESTURE RECOGNITION DATASET AUDIT - SINGLE CELL VERSION
# ============================================================
# This script will:
# 1. Scan all RGB and NIR videos
# 2. Extract metadata from folder/file names
# 3. Compute dataset statistics
# 4. Check RGB-NIR pairing
# 5. Check fps, duration, frame count, resolution
# 6. Visualize sample frames
# 7. Save CSV + JSON reports
#
# Dataset sources:
# RGB  -> https://drive.google.com/drive/folders/1lTyGgMgy1B8yRWTcvi4dOog_NCMd27GK
# NIR  -> https://drive.google.com/drive/folders/1r1Bh6rmwdzok6V6EQnSHX1x8bVSCxUgO
#
# The notebook syncs these folders into:
# data/raw/RGB/
# data/raw/NIR/
#
# Output files:
# data/processed/manifests/dataset_audit_raw_manifest.csv
# data/processed/manifests/paired_manifest_from_audit.csv
# data/reports/dataset_audit_summary.json
# ============================================================

import os
import re
import cv2
import sys
import json
import random
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================
def resolve_project_root(start: Path):
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    return start.resolve()

ROOT = resolve_project_root(Path.cwd())

RGB_SOURCE = "https://drive.google.com/drive/folders/1lTyGgMgy1B8yRWTcvi4dOog_NCMd27GK"
NIR_SOURCE = "https://drive.google.com/drive/folders/1r1Bh6rmwdzok6V6EQnSHX1x8bVSCxUgO"

LOCAL_RGB_ROOT = ROOT / "data" / "raw" / "RGB"
LOCAL_NIR_ROOT = ROOT / "data" / "raw" / "NIR"
REPORT_DIR = ROOT / "data" / "reports"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

LOCAL_RGB_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_NIR_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

def is_drive_folder(source):
    return isinstance(source, str) and source.startswith("https://drive.google.com/drive/folders/")

def sync_drive_folder(source_url: str, dest_dir: Path, label: str):
    existing_files = list(dest_dir.rglob("*.mp4"))
    if existing_files:
        print(f"{label}: using cached local files in {dest_dir}")
        return dest_dir

    cmd = [
        sys.executable, "-m", "gdown",
        "--folder", "--continue", "--remaining-ok",
        source_url,
        "-O", str(dest_dir),
    ]
    print(f"{label}: syncing from Google Drive -> {source_url}")
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if result.stdout.strip():
            print(result.stdout.strip())
    except FileNotFoundError:
        print(f"Warning: gdown is not installed in this kernel for {label}. Install it with: {sys.executable} -m pip install gdown")
    except subprocess.CalledProcessError as exc:
        err_text = (exc.stderr or exc.stdout or "").strip().splitlines()
        err_summary = err_text[-1] if err_text else "Google Drive access failed."
        print(f"Warning: failed to sync {label} from Google Drive (exit code {exc.returncode}): {err_summary}")
        print(f"The notebook will continue with any files already present in {dest_dir}.")
    return dest_dir

def resolve_dataset_root(source, local_dir: Path, label: str):
    if isinstance(source, Path):
        return source
    if isinstance(source, str) and not source.startswith("http"):
        src_path = Path(source)
        return src_path if src_path.is_absolute() else ROOT / src_path
    if is_drive_folder(source):
        return sync_drive_folder(source, local_dir, label)
    return local_dir

RGB_ROOT = resolve_dataset_root(RGB_SOURCE, LOCAL_RGB_ROOT, "RGB")
NIR_ROOT = resolve_dataset_root(NIR_SOURCE, LOCAL_NIR_ROOT, "NIR")

print("=" * 80)
print("ROOT:", ROOT)
print("RGB_ROOT exists:", RGB_ROOT.exists(), "|", RGB_ROOT)
print("NIR_ROOT exists:", NIR_ROOT.exists(), "|", NIR_ROOT)
print("REPORT_DIR:", REPORT_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("=" * 80)

# ============================================================
# 2. SUPPORTED VIDEO EXTENSIONS
# ============================================================
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV", ".MKV"}

# ============================================================
# 3. HELPER: LIST ALL VIDEO FILES
# ============================================================
def list_videos(root: Path):
    files = []
    if not root.exists():
        return files
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in VIDEO_EXTS:
            files.append(p)
    return sorted(files)

rgb_videos = list_videos(RGB_ROOT)
nir_videos = list_videos(NIR_ROOT)

print(f"RGB videos found : {len(rgb_videos)}")
print(f"NIR videos found : {len(nir_videos)}")
print(f"Total videos     : {len(rgb_videos) + len(nir_videos)}")

print("\nSample RGB paths:")
for p in rgb_videos[:10]:
    print(" ", p)

print("\nSample NIR paths:")
for p in nir_videos[:10]:
    print(" ", p)

# ============================================================
# 4. METADATA PARSING HELPERS
# ============================================================
DISTANCE_PATTERNS = [
    r"(\d+)\s*[_-]?\s*feet",
    r"(\d+)\s*[_-]?\s*ft",
    r"(\d+)\s*feet",
    r"(\d+)\s*ft",
]

KNOWN_GESTURES = {
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
    "sitdown",
    "standup",
}

def normalize_gesture(name: str):
    x = name.strip().lower().replace(" ", "_").replace("-", "_")
    if x == "sitdown":
        x = "sit_down"
    if x == "standup":
        x = "stand_up"
    return x

def extract_distance(text: str):
    txt = text.lower().replace("-", "_")
    for pat in DISTANCE_PATTERNS:
        m = re.search(pat, txt)
        if m:
            return f"{m.group(1)}_feet"
    return None

def extract_subject_id(text: str):
    """
    Tries to extract subject ID from filename/path.
    Examples supported:
    subject_001, subj12, sub_05, s01, person12, p9, id_34
    """
    txt = text.lower()
    patterns = [
        r"subject[_-]?(\d+)",
        r"subj(?:ect)?[_-]?(\d+)",
        r"\bsub[_-]?(\d+)\b",
        r"\bs[_-]?(\d{1,4})\b",
        r"person[_-]?(\d+)",
        r"\bp[_-]?(\d{1,4})\b",
        r"id[_-]?(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, txt)
        if m:
            return f"S{int(m.group(1)):03d}"
    return None

def extract_capture_id(text: str):
    """
    Tries to extract trial/take/capture ID.
    Also supports timestamp-style names such as:
    video_20260304_003212.mp4
    """
    txt = text.lower()
    patterns = [
        (r"video[_-]?(\d{8})[_-]?(\d{6})", lambda m: f"TS_{m.group(1)}{m.group(2)}"),
        (r"\b(\d{8})(\d{6})(?:\d{1,3})?\b", lambda m: f"TS_{m.group(1)}{m.group(2)}"),
        (r"take[_-]?(\d+)", lambda m: f"C{int(m.group(1)):02d}"),
        (r"clip[_-]?(\d+)", lambda m: f"C{int(m.group(1)):02d}"),
        (r"sample[_-]?(\d+)", lambda m: f"C{int(m.group(1)):02d}"),
        (r"trial[_-]?(\d+)", lambda m: f"C{int(m.group(1)):02d}"),
        (r"rep[_-]?(\d+)", lambda m: f"C{int(m.group(1)):02d}"),
        (r"\bc[_-]?(\d{1,3})\b", lambda m: f"C{int(m.group(1)):02d}"),
    ]
    for pat, formatter in patterns:
        m = re.search(pat, txt)
        if m:
            return formatter(m)
    return None

def extract_gesture_from_parts(parts):
    for part in parts:
        g = normalize_gesture(part)
        if g in KNOWN_GESTURES:
            return g
    return None

def parse_path_metadata(video_path: Path, modality: str):
    rel_str = str(video_path).lower()
    file_stem = video_path.stem

    distance = None
    for part in video_path.parts:
        distance = extract_distance(part)
        if distance:
            break

    gesture = extract_gesture_from_parts(video_path.parts)
    subject_id = extract_subject_id(rel_str)
    capture_id = extract_capture_id(file_stem) or extract_capture_id(rel_str)

    return {
        "modality": modality,
        "distance": distance,
        "gesture": gesture,
        "subject_id": subject_id,
        "capture_id": capture_id,
        "file_stem": file_stem,
    }

# ============================================================
# 5. VIDEO PROPERTY EXTRACTION
# ============================================================
def get_video_info(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return {
            "fps": np.nan,
            "num_frames": np.nan,
            "width": np.nan,
            "height": np.nan,
            "duration_sec": np.nan,
            "is_readable": False,
        }

    fps = cap.get(cv2.CAP_PROP_FPS)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

    if fps and fps > 0 and num_frames >= 0:
        duration_sec = num_frames / fps
    else:
        duration_sec = np.nan

    cap.release()

    return {
        "fps": float(fps) if fps is not None else np.nan,
        "num_frames": int(num_frames) if not np.isnan(num_frames) else np.nan,
        "width": int(width) if not np.isnan(width) else np.nan,
        "height": int(height) if not np.isnan(height) else np.nan,
        "duration_sec": float(duration_sec) if not np.isnan(duration_sec) else np.nan,
        "is_readable": True,
    }

# ============================================================
# 6. BUILD DATAFRAME
# ============================================================
DATAFRAME_COLUMNS = [
    "video_path", "filename", "suffix", "modality", "distance", "gesture",
    "subject_id", "capture_id", "file_stem", "fps", "num_frames", "width",
    "height", "duration_sec", "is_readable", "sample_id"
]

def build_video_dataframe(video_paths, modality):
    rows = []

    for vp in tqdm(video_paths, desc=f"Scanning {modality} videos"):
        meta = parse_path_metadata(vp, modality)
        info = get_video_info(vp)

        rows.append({
            "video_path": str(vp),
            "filename": vp.name,
            "suffix": vp.suffix,
            **meta,
            **info,
        })

    df = pd.DataFrame(rows, columns=DATAFRAME_COLUMNS[:-1])

    def make_sample_id(row):
        return "__".join([
            str(row.get("subject_id", "UNK_SUB")),
            str(row.get("gesture", "UNK_GES")),
            str(row.get("distance", "UNK_DIST")),
            str(row.get("capture_id", row.get("file_stem", "UNK_CAP")))
        ])

    if len(df) > 0:
        df["sample_id"] = df.apply(make_sample_id, axis=1)
    else:
        df["sample_id"] = pd.Series(dtype="object")

    return df.reindex(columns=DATAFRAME_COLUMNS)

df_rgb = build_video_dataframe(rgb_videos, "RGB")
df_nir = build_video_dataframe(nir_videos, "NIR")
df_all = pd.concat([df_rgb, df_nir], ignore_index=True)

print("\nAll dataframe shape:", df_all.shape)
display(df_all.head())

# ============================================================
# 7. SAVE RAW MANIFEST
# ============================================================
raw_manifest_path = MANIFEST_DIR / "dataset_audit_raw_manifest.csv"
df_all.to_csv(raw_manifest_path, index=False)
print("Saved raw manifest to:", raw_manifest_path)

# ============================================================
# 8. BASIC COUNTS
# ============================================================
print("\n" + "=" * 80)
print("BASIC COUNTS")
print("=" * 80)
print("How many total videos? ", len(df_all))
print("How many RGB videos?   ", (df_all["modality"] == "RGB").sum() if len(df_all) else 0)
print("How many NIR videos?   ", (df_all["modality"] == "NIR").sum() if len(df_all) else 0)
print("How many unique subjects? ", df_all["subject_id"].dropna().nunique() if len(df_all) else 0)
print("How many unique gestures? ", df_all["gesture"].dropna().nunique() if len(df_all) else 0)
print("How many unique distances?", df_all["distance"].dropna().nunique() if len(df_all) else 0)

# ============================================================
# 9. MISSING METADATA SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("MISSING METADATA SUMMARY")
print("=" * 80)
missing_summary = pd.DataFrame({
    "missing_count": df_all[["distance", "gesture", "subject_id", "capture_id"]].isna().sum(),
    "missing_percent": (df_all[["distance", "gesture", "subject_id", "capture_id"]].isna().mean() * 100).round(2)
}) if len(df_all) else pd.DataFrame()

display(missing_summary)

problem_rows = df_all[
    df_all["distance"].isna() |
    df_all["gesture"].isna() |
    df_all["subject_id"].isna()
].copy() if len(df_all) else pd.DataFrame()

print("Rows with important missing metadata:", len(problem_rows))
if len(problem_rows):
    display(problem_rows[["video_path", "distance", "gesture", "subject_id", "capture_id"]].head(50))

# ============================================================
# 10. COUNT TABLES
# ============================================================
print("\n" + "=" * 80)
print("COUNT TABLES")
print("=" * 80)

count_by_modality = df_all["modality"].value_counts().rename_axis("modality").reset_index(name="count") if len(df_all) else pd.DataFrame()
count_by_gesture = (
    df_all.groupby(["modality", "gesture"])
    .size()
    .reset_index(name="count")
    .sort_values(["modality", "gesture"])
) if len(df_all) else pd.DataFrame()

count_by_distance = (
    df_all.groupby(["modality", "distance"])
    .size()
    .reset_index(name="count")
    .sort_values(["modality", "distance"])
) if len(df_all) else pd.DataFrame()

count_by_subject = (
    df_all.groupby(["modality", "subject_id"])
    .size()
    .reset_index(name="count")
    .sort_values(["modality", "subject_id"])
) if len(df_all) else pd.DataFrame()

gesture_distance_table = pd.crosstab(
    index=[df_all["modality"], df_all["gesture"]],
    columns=df_all["distance"],
    margins=True
) if len(df_all) else pd.DataFrame()

print("\nCount by modality")
display(count_by_modality)

print("\nCount by gesture")
display(count_by_gesture)

print("\nCount by distance")
display(count_by_distance)

print("\nCount by subject")
display(count_by_subject.head(30))

print("\nGesture × Distance Crosstab")
display(gesture_distance_table)

# ============================================================
# 11. BROKEN FILES / VIDEO PROPERTY STATS
# ============================================================
print("\n" + "=" * 80)
print("VIDEO HEALTH CHECK")
print("=" * 80)

broken_files = df_all[df_all["is_readable"].fillna(False) == False].copy() if len(df_all) else pd.DataFrame()
print("Unreadable videos:", len(broken_files))
if len(broken_files):
    display(broken_files[["video_path", "modality"]].head(50))

duration_stats = df_all.groupby("modality")["duration_sec"].describe().round(3) if len(df_all) else pd.DataFrame()
fps_stats = df_all.groupby("modality")["fps"].describe().round(3) if len(df_all) else pd.DataFrame()

resolution_stats = (
    df_all.groupby(["modality", "width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values(["modality", "count"], ascending=[True, False])
) if len(df_all) else pd.DataFrame()

print("\nDuration stats")
display(duration_stats)

print("\nFPS stats")
display(fps_stats)

print("\nResolution stats")
display(resolution_stats.head(20))

duration_check = df_all.copy()
if len(duration_check):
    duration_check["duration_error_from_5s"] = (duration_check["duration_sec"] - 5.0).abs()
    print("Videos with > 0.5 sec deviation from 5 sec:",
          (duration_check["duration_error_from_5s"] > 0.5).sum())
    display(
        duration_check[duration_check["duration_error_from_5s"] > 0.5][
            ["video_path", "modality", "duration_sec", "fps", "num_frames"]
        ].head(50)
    )

# ============================================================
# 12. FILE NAMING CONSISTENCY CHECK
# ============================================================
print("\n" + "=" * 80)
print("FILE NAMING CONSISTENCY")
print("=" * 80)

def filename_pattern_signature(name):
    x = name.lower()
    x = re.sub(r"\d+", "<NUM>", x)
    return x

if len(df_all):
    df_all["filename_signature"] = df_all["filename"].apply(filename_pattern_signature)
    signature_counts = df_all["filename_signature"].value_counts().reset_index()
    signature_counts.columns = ["filename_signature", "count"]
else:
    signature_counts = pd.DataFrame()

display(signature_counts.head(20))

# ============================================================
# 13. BUILD PAIR KEYS
# ============================================================
def build_pair_key(df):
    out = df.copy()
    for col in ["subject_id", "gesture", "distance", "capture_id", "file_stem"]:
        if col not in out.columns:
            out[col] = pd.Series(dtype="object")

    capture_key = out["capture_id"].where(out["capture_id"].notna(), out["file_stem"])
    use_subject = out["subject_id"].notna().any()

    if use_subject:
        out["pair_key_strong"] = (
            out["subject_id"].fillna("UNK") + "__" +
            out["gesture"].fillna("UNK") + "__" +
            out["distance"].fillna("UNK") + "__" +
            capture_key.fillna("UNK")
        )
        out["pair_key_weak"] = (
            out["subject_id"].fillna("UNK") + "__" +
            out["gesture"].fillna("UNK") + "__" +
            out["distance"].fillna("UNK")
        )
    else:
        out["pair_key_strong"] = (
            out["gesture"].fillna("UNK") + "__" +
            out["distance"].fillna("UNK") + "__" +
            capture_key.fillna("UNK")
        )
        out["pair_key_weak"] = (
            out["gesture"].fillna("UNK") + "__" +
            out["distance"].fillna("UNK")
        )
    return out

df_all = build_pair_key(df_all)
df_rgb = df_all[df_all["modality"] == "RGB"].copy()
df_nir = df_all[df_all["modality"] == "NIR"].copy()

# ============================================================
# 14. PAIRING CHECK
# ============================================================
print("\n" + "=" * 80)
print("PAIRING CHECK")
print("=" * 80)

rgb_keys = set(df_rgb["pair_key_strong"]) if len(df_rgb) else set()
nir_keys = set(df_nir["pair_key_strong"]) if len(df_nir) else set()

common_keys = rgb_keys.intersection(nir_keys)
rgb_only_keys = rgb_keys - nir_keys
nir_only_keys = nir_keys - rgb_keys

print("Common pair keys :", len(common_keys))
print("RGB-only keys    :", len(rgb_only_keys))
print("NIR-only keys    :", len(nir_only_keys))

missing_rgb_pairs = df_rgb[df_rgb["pair_key_strong"].isin(rgb_only_keys)].copy() if len(df_rgb) else pd.DataFrame()
missing_nir_pairs = df_nir[df_nir["pair_key_strong"].isin(nir_only_keys)].copy() if len(df_nir) else pd.DataFrame()

print("\nRGB samples without NIR pair:", len(missing_rgb_pairs))
if len(missing_rgb_pairs):
    display(missing_rgb_pairs[["video_path", "subject_id", "gesture", "distance", "capture_id"]].head(30))

print("\nNIR samples without RGB pair:", len(missing_nir_pairs))
if len(missing_nir_pairs):
    display(missing_nir_pairs[["video_path", "subject_id", "gesture", "distance", "capture_id"]].head(30))

# ============================================================
# 15. BUILD PAIRED DATAFRAME
# ============================================================
rgb_pair_df = df_rgb[[
    "pair_key_strong", "pair_key_weak",
    "subject_id", "gesture", "distance", "capture_id",
    "video_path", "fps", "num_frames", "duration_sec", "width", "height"
]].copy() if len(df_rgb) else pd.DataFrame(columns=[
    "pair_key_strong", "pair_key_weak", "subject_id", "gesture", "distance", "capture_id",
    "video_path", "fps", "num_frames", "duration_sec", "width", "height"
])

rgb_pair_df = rgb_pair_df.rename(columns={
    "pair_key_weak": "rgb_pair_key_weak",
    "subject_id": "rgb_subject_id",
    "gesture": "rgb_gesture",
    "distance": "rgb_distance",
    "capture_id": "rgb_capture_id",
    "video_path": "rgb_video_path",
    "fps": "rgb_fps",
    "num_frames": "rgb_num_frames",
    "duration_sec": "rgb_duration_sec",
    "width": "rgb_width",
    "height": "rgb_height",
})

nir_pair_df = df_nir[[
    "pair_key_strong", "pair_key_weak",
    "subject_id", "gesture", "distance", "capture_id",
    "video_path", "fps", "num_frames", "duration_sec", "width", "height"
]].copy() if len(df_nir) else pd.DataFrame(columns=[
    "pair_key_strong", "pair_key_weak", "subject_id", "gesture", "distance", "capture_id",
    "video_path", "fps", "num_frames", "duration_sec", "width", "height"
])

nir_pair_df = nir_pair_df.rename(columns={
    "pair_key_weak": "nir_pair_key_weak",
    "subject_id": "nir_subject_id",
    "gesture": "nir_gesture",
    "distance": "nir_distance",
    "capture_id": "nir_capture_id",
    "video_path": "nir_video_path",
    "fps": "nir_fps",
    "num_frames": "nir_num_frames",
    "duration_sec": "nir_duration_sec",
    "width": "nir_width",
    "height": "nir_height",
})

paired_df = pd.merge(
    rgb_pair_df,
    nir_pair_df,
    on=["pair_key_strong"],
    how="inner"
)

if len(paired_df):
    paired_df["subject_id"] = paired_df["rgb_subject_id"].combine_first(paired_df["nir_subject_id"])
    paired_df["gesture"] = paired_df["rgb_gesture"].combine_first(paired_df["nir_gesture"])
    paired_df["distance"] = paired_df["rgb_distance"].combine_first(paired_df["nir_distance"])
    paired_df["capture_id"] = paired_df["rgb_capture_id"].combine_first(paired_df["nir_capture_id"])
    paired_df["pair_key_weak"] = paired_df["rgb_pair_key_weak"].combine_first(paired_df["nir_pair_key_weak"])
else:
    paired_df["subject_id"] = pd.Series(dtype="object")
    paired_df["gesture"] = pd.Series(dtype="object")
    paired_df["distance"] = pd.Series(dtype="object")
    paired_df["capture_id"] = pd.Series(dtype="object")
    paired_df["pair_key_weak"] = pd.Series(dtype="object")

print("Paired samples:", len(paired_df))
display(paired_df.head())

# ============================================================
# 16. PAIR CONSISTENCY CHECKS
# ============================================================
if len(paired_df):
    paired_df["fps_diff"] = (paired_df["rgb_fps"] - paired_df["nir_fps"]).abs()
    paired_df["duration_diff"] = (paired_df["rgb_duration_sec"] - paired_df["nir_duration_sec"]).abs()
    paired_df["frame_diff"] = (paired_df["rgb_num_frames"] - paired_df["nir_num_frames"]).abs()

    pair_consistency = {
        "mean_fps_diff": float(paired_df["fps_diff"].mean()),
        "max_fps_diff": float(paired_df["fps_diff"].max()),
        "mean_duration_diff": float(paired_df["duration_diff"].mean()),
        "max_duration_diff": float(paired_df["duration_diff"].max()),
        "mean_frame_diff": float(paired_df["frame_diff"].mean()),
        "max_frame_diff": float(paired_df["frame_diff"].max()),
    }
else:
    paired_df["fps_diff"] = []
    paired_df["duration_diff"] = []
    paired_df["frame_diff"] = []
    pair_consistency = {}

print("\nPair consistency:")
print(json.dumps(pair_consistency, indent=4))

suspicious_pairs = paired_df[
    (paired_df["duration_diff"] > 0.3) |
    (paired_df["fps_diff"] > 1.0) |
    (paired_df["frame_diff"] > 10)
].copy() if len(paired_df) else pd.DataFrame()

print("\nSuspicious paired samples:", len(suspicious_pairs))
if len(suspicious_pairs):
    display(
        suspicious_pairs[[
            "subject_id", "gesture", "distance", "capture_id",
            "rgb_video_path", "nir_video_path",
            "rgb_fps", "nir_fps",
            "rgb_num_frames", "nir_num_frames",
            "rgb_duration_sec", "nir_duration_sec"
        ]].head(30)
    )

# ============================================================
# 17. SAVE PAIRED MANIFEST
# ============================================================
paired_manifest_path = MANIFEST_DIR / "paired_manifest_from_audit.csv"
paired_df.to_csv(paired_manifest_path, index=False)
print("Saved paired manifest to:", paired_manifest_path)

# ============================================================
# 18. PLOTS
# ============================================================
if len(count_by_gesture):
    plt.figure(figsize=(10, 5))
    plot_df = count_by_gesture.copy()
    for modality in plot_df["modality"].unique():
        sub = plot_df[plot_df["modality"] == modality]
        plt.bar(
            [f"{g}\n({modality})" for g in sub["gesture"]],
            sub["count"],
            alpha=0.7,
            label=modality
        )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.title("Video count by gesture and modality")
    plt.legend()
    plt.tight_layout()
    plt.show()

if len(count_by_distance):
    plt.figure(figsize=(8, 4))
    plot_df = count_by_distance.copy()
    for modality in plot_df["modality"].unique():
        sub = plot_df[plot_df["modality"] == modality]
        plt.bar(
            [f"{d}\n({modality})" for d in sub["distance"]],
            sub["count"],
            alpha=0.7,
            label=modality
        )
    plt.ylabel("Count")
    plt.title("Video count by distance and modality")
    plt.legend()
    plt.tight_layout()
    plt.show()

# ============================================================
# 19. FRAME READER FOR VISUALIZATION
# ============================================================
def read_frame_at(video_path, frame_index=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None

    if frame_index is None:
        frame_index = total_frames // 2

    frame_index = max(0, min(frame_index, total_frames - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        return None

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return frame

# ============================================================
# 20. VISUALIZE RANDOM PAIRED SAMPLES
# ============================================================
def show_random_pairs(paired_df, n=6, seed=42):
    if len(paired_df) == 0:
        print("No paired samples found.")
        return

    sample_rows = paired_df.sample(min(n, len(paired_df)), random_state=seed)

    fig, axes = plt.subplots(len(sample_rows), 2, figsize=(10, 4 * len(sample_rows)))
    if len(sample_rows) == 1:
        axes = np.array([axes])

    for i, (_, row) in enumerate(sample_rows.iterrows()):
        rgb_frame = read_frame_at(row["rgb_video_path"])
        nir_frame = read_frame_at(row["nir_video_path"])

        axes[i, 0].imshow(rgb_frame if rgb_frame is not None else np.zeros((100, 100, 3), dtype=np.uint8))
        axes[i, 0].set_title(f"RGB | {row['gesture']} | {row['distance']} | {row['subject_id']}")
        axes[i, 0].axis("off")

        if nir_frame is not None:
            axes[i, 1].imshow(nir_frame, cmap="gray" if nir_frame.ndim == 2 else None)
        else:
            axes[i, 1].imshow(np.zeros((100, 100, 3), dtype=np.uint8))
        axes[i, 1].set_title(f"NIR | {row['gesture']} | {row['distance']} | {row['subject_id']}")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()

# ============================================================
# 21. VISUALIZE BY GESTURE AND DISTANCE
# ============================================================
def show_examples_by_gesture_distance(paired_df, gestures=None, distances=None, samples_per_group=2):
    temp = paired_df.copy()

    if gestures is not None:
        temp = temp[temp["gesture"].isin(gestures)]
    if distances is not None:
        temp = temp[temp["distance"].isin(distances)]

    groups = temp.groupby(["gesture", "distance"])
    rows = []

    for (gesture, distance), grp in groups:
        grp = grp.sample(min(samples_per_group, len(grp)), random_state=42)
        for _, row in grp.iterrows():
            rows.append((gesture, distance, row))

    if len(rows) == 0:
        print("No matching samples found.")
        return

    fig, axes = plt.subplots(len(rows), 2, figsize=(10, 4 * len(rows)))
    if len(rows) == 1:
        axes = np.array([axes])

    for i, (gesture, distance, row) in enumerate(rows):
        rgb_frame = read_frame_at(row["rgb_video_path"])
        nir_frame = read_frame_at(row["nir_video_path"])

        axes[i, 0].imshow(rgb_frame if rgb_frame is not None else np.zeros((100, 100, 3), dtype=np.uint8))
        axes[i, 0].set_title(f"RGB | {gesture} | {distance} | {row['subject_id']}")
        axes[i, 0].axis("off")

        if nir_frame is not None:
            axes[i, 1].imshow(nir_frame, cmap="gray" if nir_frame.ndim == 2 else None)
        else:
            axes[i, 1].imshow(np.zeros((100, 100, 3), dtype=np.uint8))
        axes[i, 1].set_title(f"NIR | {gesture} | {distance} | {row['subject_id']}")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()

print("\nShowing random paired samples...")
show_random_pairs(paired_df, n=6)

print("\nShowing structured examples by gesture and distance...")
show_examples_by_gesture_distance(
    paired_df,
    gestures=["doctor", "emergency", "fire"],
    distances=["4_feet", "8_feet"],
    samples_per_group=2
)

# ============================================================
# 22. SAVE FINAL AUDIT SUMMARY
# ============================================================
audit_summary = {
    "total_videos": int(len(df_all)),
    "rgb_videos": int((df_all["modality"] == "RGB").sum()) if len(df_all) else 0,
    "nir_videos": int((df_all["modality"] == "NIR").sum()) if len(df_all) else 0,
    "unique_subjects": int(df_all["subject_id"].dropna().nunique()) if len(df_all) else 0,
    "unique_gestures": int(df_all["gesture"].dropna().nunique()) if len(df_all) else 0,
    "unique_distances": int(df_all["distance"].dropna().nunique()) if len(df_all) else 0,
    "unreadable_videos": int((df_all["is_readable"].fillna(False) == False).sum()) if len(df_all) else 0,
    "paired_samples": int(len(paired_df)),
    "rgb_without_pair": int(len(missing_rgb_pairs)),
    "nir_without_pair": int(len(missing_nir_pairs)),
    "mean_rgb_duration": float(df_rgb["duration_sec"].mean()) if len(df_rgb) else None,
    "mean_nir_duration": float(df_nir["duration_sec"].mean()) if len(df_nir) else None,
    "mean_rgb_fps": float(df_rgb["fps"].mean()) if len(df_rgb) else None,
    "mean_nir_fps": float(df_nir["fps"].mean()) if len(df_nir) else None,
}

summary_path = REPORT_DIR / "dataset_audit_summary.json"
with open(summary_path, "w") as f:
    json.dump(audit_summary, f, indent=4)

print("\n" + "=" * 80)
print("FINAL AUDIT SUMMARY")
print("=" * 80)
print(json.dumps(audit_summary, indent=4))
print("Saved summary to:", summary_path)

# ============================================================
# 23. FINAL CONCLUSION
# ============================================================
print("\n" + "=" * 80)
print("NEXT THINGS TO CHECK MANUALLY")
print("=" * 80)
print("""
1. Are subject_id values extracted correctly?
2. Are gesture names correct and consistent?
3. Are distance labels detected correctly?
4. How many RGB and NIR videos are truly paired?
5. Are there missing files or broken files?
6. Are durations near 5 sec?
7. Are fps values similar between RGB and NIR?
8. Is naming consistent enough for subject-wise splits?

If subject_id or capture_id are missing badly, fix file naming before training any model.
If pairing is weak, do not start fusion yet.
First repair metadata and create a clean paired manifest.
""")